# 🎯 Answer Accuracy Evaluation — Showcase Notebook

This notebook is a **portfolio-grade demonstration** of GenAI answer-accuracy evaluation. It is designed to show not only that a model can be scored, but that the evaluation is thoughtfully framed, reproducible, interpretable, and extensible.

The notebook answers six questions:

1. **What model are we testing?** Target model and provider are confirmed before the run.
2. **What datasets are we testing on?** Dataset preset, selected datasets, splits, cache paths, and sample sizes are shown explicitly.
3. **What kind of tasks are these?** Different datasets have different structures: multiple-choice, open-domain QA, context-grounded QA, multi-hop QA, truthfulness-style generation.
4. **What metrics are used?** Exact match, contains match, token F1, semantic similarity, and composite score are explained with limitations.
5. **Where did the model struggle?** Diagnostics flag weak, borderline, and metric-disagreement cases with reasons.
6. **What can be reported?** An in-notebook HTML executive report summarizes target model, datasets, pass rate, flagged cases, and artifacts.

Heavy implementation lives in `src/genai_capability_bench`; the notebook is the guided evaluation and communication layer.


---

## 🧭 1. Evaluation Framing

**Capability under test:** closed-book answer accuracy.

Answer accuracy tests whether a model can produce an expected factual answer. It is one of the most basic LLM capabilities, but it has important design choices:

| Design Question | Why It Matters |
|---|---|
| Closed-book or context-grounded? | Closed-book tests model knowledge; context-grounded tests reading and grounding. |
| Short answer or multiple choice? | Short answer needs reference/semantic matching; multiple choice should use exact option accuracy. |
| Public benchmark or custom golden set? | Public datasets support comparability; custom sets support business relevance. |
| Rule-based metric or judge model? | Rule-based metrics are reproducible; judge models help with ambiguity but add cost and variability. |

This notebook starts with transparent rule-based metrics and supports optional judge review only for flagged cases.


---

## 🗃️ 2. Benchmark Dataset Strategy

This notebook treats dataset selection as a benchmark-design decision, not a file-loading detail. The recommended default is `curated_knowledge`, a repo-native benchmark assembled from compatible public benchmark caches.

### Curated Knowledge QA v1

`curated_knowledge_v1` is constructed as follows:

1. Public benchmark rows are first normalized into the shared `EvalTask` schema.
2. Compatible closed-book answer-accuracy rows are copied from the largest available local normalized caches.
3. The curation step preserves the original source question, expected answer, reference answers, and task text.
4. Only compatible standardized metadata is added: broad category, subcategory, source dataset, source task ID, source cache path, scoring profile, and integrity notes.
5. Natural Questions, SQuAD, HotpotQA, and TruthfulQA are intentionally excluded from this default closed-book set because their task shapes are better suited to long-reference QA, RAG/reasoning, or truthfulness workflows.

Current curated sources:

| Source | Role | Rows |
|---|---|---:|
| MMLU | Broad academic and professional multiple-choice knowledge | 14,042 |
| TriviaQA | Concise open-domain factual QA | 17,944 |
| ARC Challenge | Science exam-style multiple-choice QA | 1,170 |
| **Total** | Versioned local curated benchmark | **33,156** |

### Presets

| Preset | Datasets | Primary Question |
|---|---|---|
| `curated_knowledge` | `curated_knowledge_v1` | Recommended default: broad source-preserving closed-book knowledge benchmark. |
| `local_smoke` | `answer_accuracy_sample` | Does the workflow run correctly with a small local golden set? |
| `broad_knowledge` | `mmlu` | How does the model perform on the original MMLU benchmark shape? |
| `open_domain_qa` | `triviaqa`, `natural_questions` | Can the model answer open-domain factual questions with acceptable reference alignment? |
| `science_exam` | `arc` | Can the model handle science exam-style factual reasoning? |
| `custom_golden_set` | `custom` | How does the model perform on organization-specific questions that matter in practice? |

Operational guidance:

- Use `curated_knowledge` for the main showcase run.
- The default `SAMPLE_STRATEGY='auto'` resolves to stratified sampling for the curated dataset: source dataset first, then broad category within source.
- Use `quick_check` to validate credentials and mechanics; use `report_sample` for a more credible sample report.
- `RUN_MODE='full_public'` evaluates the full selected split/file. Use only with explicit budget and runtime approval.
- Keep `DATASET_KEYS` explicit when you need full control; presets are convenient benchmark portfolios.


---

## 📏 3. Metric Standards and Interpretation

This repo uses **dataset-aware scoring profiles** rather than one universal score. The same metric is not equally valid for every task shape.

Core principles:

- **Short-answer QA**: Exact Match and Token F1 are primary; semantic similarity, BLEU, and ROUGE-L are secondary diagnostics.
- **Multiple choice**: exact option/answer match is primary; free-form overlap metrics are secondary.
- **Long-reference QA**: ROUGE-L and semantic similarity become more relevant, but low scores may indicate that the dataset reference is a long passage while the model gave a concise answer.
- **Contains Match**: useful diagnostic, but no longer a primary full-credit score because it can over-credit wrong long answers.
- **LLM judge**: reserved for ambiguous, open-ended, or high-stakes review, not the default deterministic score.
- **Embedding model**: configured from `.env` for optional API-based semantic similarity. The default run uses local TF-IDF for reproducibility unless `USE_API_EMBEDDINGS=True`.

The notebook displays the repo-wide metric standards and scoring profiles below. The selected dataset's `scoring_profile` determines the primary score used for pass/fail. Some profiles use a composite score, and the formula is shown explicitly in the scoring profile table.

Current primary-score formulas:

| Scoring Profile | Primary Score Formula |
|---|---|
| `short_answer_qa` | `max(exact_match, 0.65 * token_f1 + 0.35 * semantic_similarity)` |
| `multiple_choice` | exact match against answer text or displayed option label |
| `long_reference_qa` | `max(0.55 * rouge_l + 0.45 * semantic_similarity, 0.50 * token_f1 + 0.50 * semantic_similarity)` |
| `truthfulness_generation` | judge rubric score, with deterministic metrics as supporting evidence |

Interpretation rule of thumb:

| Score | Interpretation |
|---|---|
| 0.90 - 1.00 | Strong match for the dataset's scoring profile |
| 0.70 - 0.89 | Likely acceptable; review in high-stakes settings |
| 0.40 - 0.69 | Partial/uncertain; review recommended |
| 0.00 - 0.39 | Likely incorrect, unsupported, or reference-shape mismatch |


In [ ]:
# =============================================================================
# CELL 1: IMPORTS & SETUP
# =============================================================================
from pathlib import Path
import sys
import warnings

warnings.filterwarnings('ignore')

REPO_ROOT = Path('..').resolve()
sys.path.insert(0, str(REPO_ROOT / 'src'))

from genai_capability_bench.notebooks.answer_accuracy import (
    ANSWER_ACCURACY_DATASET_PRESETS,
    RUN_PROFILES,
    default_judge_model_name,
    display_answer_accuracy_visuals,
    display_configuration_summary,
    display_curated_dataset_preview,
    display_executive_report,
    display_model_context,
    display_notebook_ready,
    display_preflight_check,
    display_reference_catalogs,
    display_result_diagnostics,
    display_run_completion,
    display_saved_artifacts,
    prepare_answer_accuracy_plan,
)
from genai_capability_bench.workflows.answer_accuracy import run_answer_accuracy_workflow

OUTPUT_ROOT = REPO_ROOT / 'outputs' / 'runs'
display_notebook_ready(REPO_ROOT)


In [ ]:
# =============================================================================
# CELL 2: CATALOGS — DATASETS, SAMPLE SIZE, AND METRICS
# =============================================================================
display_reference_catalogs()
display_curated_dataset_preview(REPO_ROOT)

---

## ⚙️ 4. Configuration — Benchmark Controls

This is the main control surface for the run. A non-technical reviewer should be able to read this section and understand **what model**, **what datasets**, **how many samples**, and **what review policy** are being used.

### Model Config

- `eval_openai_compatible_template.yaml` reads provider settings from `.env` and is intended for Azure OpenAI, OpenAI, Ollama, Groq, Together AI, LM Studio, and other OpenAI-compatible endpoints.
- `eval_core_demo.yaml` runs the local mock model and is useful for validating the workflow without API cost.
- Some GPT-5/reasoning deployments reject `temperature`, `max_tokens`, or similar generation controls. Set `OPENAI_TEMPERATURE=omit`, `OPENAI_MAX_TOKENS=omit`, and `OPENAI_TOKEN_PARAMETER=` in `.env`, or leave those variables undefined when using this template.

### Dataset and Run Size Controls

Most users only need to choose two things:

- `DATASET_PRESET`: the benchmark portfolio. The recommended default is `curated_knowledge`; alternatives include `local_smoke`, `broad_knowledge`, `open_domain_qa`, `science_exam`, or `knowledge_portfolio`.
- `RUN_MODE`: the run size and safety posture. Use `quick_check`, `exploratory`, `report_sample`, or `full_public`.

How to interpret the derived settings:

- **Sample per dataset** means the notebook samples up to that many rows from each selected dataset. For `knowledge_portfolio`, `100` means up to 100 MMLU, 100 TriviaQA, 100 Natural Questions, and 100 ARC rows before model count is applied. For `curated_knowledge`, it means up to that many rows from the local curated JSONL file.
- **Sampling strategy** controls which rows are selected. `auto` is recommended: it uses source/category stratified sampling for `curated_knowledge` and regular random sampling for ordinary source benchmarks.
- **Estimated model calls** means sampled tasks × number of target models. This is the practical cost/runtime driver.
- **Safety cap** stops the run if the materialized task count is unexpectedly larger than planned.
- **Full public split** means the entire selected public benchmark split. It only happens in `RUN_MODE = 'full_public'`; otherwise full-split loading is blocked.
- **Checkpoint every** controls how often resumable progress is written to disk.
- **Auto resume** reuses the latest compatible checkpoint when dataset, model, sample settings, and scoring-method version match.
- **Checkpoint cleanup** removes legacy, corrupt, or stale method-version checkpoints. Different valid benchmark configurations are preserved but will not be reused unless their fingerprint matches.

`curated_knowledge` is the repo-native default benchmark: it uses the full available compatible closed-book caches from MMLU, TriviaQA, and ARC; preserves the original question/answer/reference text; and adds a uniform category taxonomy plus provenance metadata. Run modes sample from this larger local file to control cost. Advanced users can still override splits, categories, checkpoint paths, custom datasets, or `DATASET_KEYS` below. Multi-dataset results are reported as a portfolio: each dataset keeps its own scoring profile, then the report summarizes dataset-level evidence side by side.

### Judge Review

Judge review is disabled by default. When enabled, it is used for flagged/ambiguous cases, not as the primary score. This keeps the benchmark reproducible while still supporting expert-style review where rule-based metrics are weak.


In [ ]:
# =============================================================================
# CELL 3: CONFIGURATION BLOCK — edit this cell
# =============================================================================
# Primary choices most users should edit:
MODEL_CONFIG_PATH = REPO_ROOT / 'configs' / 'eval_openai_compatible_template.yaml'
DATASET_PRESET = 'curated_knowledge'  # Recommended default: source-preserving curated MMLU + TriviaQA + ARC benchmark.
RUN_MODE = 'report_sample'           # quick_check | exploratory | report_sample | full_public
ENABLE_JUDGE_REVIEW = True
USE_API_EMBEDDINGS = True      # Default False uses local TF-IDF; True uses OPENAI_EMBEDDING_MODEL if configured.

# Optional: manually override the preset dataset list.
# Leave as None to use ANSWER_ACCURACY_DATASET_PRESETS[DATASET_PRESET].
# Example multi-dataset override: DATASET_KEYS = ['mmlu', 'triviaqa', 'arc']
DATASET_KEYS = None

# Advanced controls. These are usually left unchanged for standard demo runs.
DATASET_SPLITS = {}                # e.g., {'mmlu': 'test'}
CUSTOM_DATASET_PATH = None         # Required only when DATASET_PRESET='custom_golden_set'.
SELECTED_CATEGORIES = 'ALL'       # For curated_knowledge, try ['history', 'science', 'general_knowledge'].
SAMPLE_STRATEGY = 'auto'          # auto => stratified for curated_knowledge; alternatives: 'random', 'stratified'.
RANDOM_SEED = 42
CHECKPOINT_EVERY = 50
RESUME_FROM_CHECKPOINT = None      # Optional manual checkpoint folder/file; auto-resume is enabled below.
AUTO_RESUME_FROM_LATEST = True
CLEANUP_INCOMPATIBLE_CHECKPOINTS = True

DOWNLOAD_IF_MISSING = True
CACHE_LOCAL_COPY = True
REFRESH_DATASET_CACHE = False

PASS_THRESHOLD = None              # None uses default_pass_threshold from the model config.
RUN_ID_PREFIX = 'answer_accuracy_demo'
JUDGE_MODEL_NAME = default_judge_model_name()
# Caps how many flagged/ambiguous cases are sent to the judge LLM.
# This controls judge-review cost/latency only; it does not limit benchmark sample size.
JUDGE_MAX_CASES = 10

# Optional UI. Keep False if your notebook frontend renders widgets as raw text.
USE_INTERACTIVE_WIDGETS = False
if USE_INTERACTIVE_WIDGETS:
    try:
        import ipywidgets as widgets
        from IPython.display import display

        preset_widget = widgets.Dropdown(options=list(ANSWER_ACCURACY_DATASET_PRESETS), value=DATASET_PRESET, description='Preset')
        mode_widget = widgets.Dropdown(options=list(RUN_PROFILES), value=RUN_MODE, description='Run mode')
        judge_widget = widgets.Checkbox(value=ENABLE_JUDGE_REVIEW, description='Judge review')
        display(widgets.VBox([preset_widget, mode_widget, judge_widget]))
        DATASET_PRESET = preset_widget.value
        RUN_MODE = mode_widget.value
        ENABLE_JUDGE_REVIEW = bool(judge_widget.value)
    except Exception as exc:
        print(f'Interactive widgets unavailable; using manual config values above. Detail: {exc}')

plan = prepare_answer_accuracy_plan(
    repo_root=REPO_ROOT,
    output_root=OUTPUT_ROOT,
    model_config_path=MODEL_CONFIG_PATH,
    dataset_preset=DATASET_PRESET,
    run_mode=RUN_MODE,
    dataset_keys=DATASET_KEYS,
    dataset_splits=DATASET_SPLITS,
    custom_dataset_path=CUSTOM_DATASET_PATH,
    selected_categories=SELECTED_CATEGORIES,
    sample_strategy=SAMPLE_STRATEGY,
    random_seed=RANDOM_SEED,
    checkpoint_every=CHECKPOINT_EVERY,
    resume_from_checkpoint=RESUME_FROM_CHECKPOINT,
    auto_resume_from_latest=AUTO_RESUME_FROM_LATEST,
    cleanup_incompatible_checkpoints=CLEANUP_INCOMPATIBLE_CHECKPOINTS,
    download_if_missing=DOWNLOAD_IF_MISSING,
    cache_local_copy=CACHE_LOCAL_COPY,
    refresh_dataset_cache=REFRESH_DATASET_CACHE,
    pass_threshold=PASS_THRESHOLD,
    run_id_prefix=RUN_ID_PREFIX,
    enable_judge_review=ENABLE_JUDGE_REVIEW,
    judge_model_name=JUDGE_MODEL_NAME,
    judge_max_cases=JUDGE_MAX_CASES,
    use_api_embeddings=USE_API_EMBEDDINGS,
)
workflow_config = plan.workflow_config
PUBLIC_MODEL_LABEL = plan.public_model_label

display_configuration_summary(plan)


In [ ]:
# =============================================================================
# CELL 4: TARGET MODEL, JUDGE, AND EMBEDDING CONTEXT
# =============================================================================
display_model_context(plan)


---

## ✅ 5. Dataset Materialization and Pre-Flight Check

This section confirms the benchmark before any model calls are made. It is intentionally explicit because dataset choice is part of the evaluation evidence.

The **Dataset Manifest** answers four practical questions:

- Which dataset key and split were selected?
- Was the source local, cached, custom, or downloaded from the public dataset source?
- How many raw tasks were loaded and how many answer-accuracy tasks were retained?
- Which normalized categories are represented in the selected sample?

If the manifest does not match your intent, stop and adjust `DATASET_PRESET`, `DATASET_KEYS`, `DATASET_SPLITS`, `SELECTED_CATEGORIES`, `SAMPLE_SIZE_PER_DATASET`, or `MAX_TASKS_PER_RUN` before running the evaluation. A full public split can be very large and is blocked unless `ALLOW_FULL_PUBLIC_DATASET=True`.


In [ ]:
# =============================================================================
# CELL 5: PRE-FLIGHT CHECK
# =============================================================================
tasks, dataset_manifest_df = display_preflight_check(plan)


---

## 🚀 6. Run Evaluation

This cell runs the answer-accuracy workflow. For real LLMs, this is where API calls begin. The progress bar shows completed model calls and percentage progress.

What gets saved:

- raw normalized results
- category summary
- dataset/category summary
- leaderboard
- diagnostics with flag reasons
- markdown report
- dataset manifest
- resumable checkpoint snapshots under `outputs/runs/_checkpoints/<run_id>/`
- automatic checkpoint pickup when the benchmark fingerprint matches
- stale checkpoint cleanup when the scoring-method version changes


In [ ]:
# =============================================================================
# CELL 6: RUN EVALUATION
# =============================================================================
run = run_answer_accuracy_workflow(workflow_config, show_progress=True)
artifact_table = display_run_completion(run, plan)


---

## 🔎 7. Result Analysis and Diagnostics

A score alone is not enough. The diagnostic table explains **what was flagged and why**.

Flag categories:

| Flag | Meaning |
|---|---|
| High review priority | Score is very low; likely incorrect or missing answer. |
| Near threshold | Score is below pass threshold but not catastrophic. |
| Metric disagreement | Metrics disagree strongly; may need semantic or judge review. |
| Looks good | No immediate issue triggered. |

If judge review is enabled, flagged cases may also include `judge_score` and `judge_reason`.


In [ ]:
# =============================================================================
# CELL 7: RESULT ANALYSIS AND DIAGNOSTICS
# =============================================================================
display_tables = display_result_diagnostics(run, plan)


---

## 📊 8. Visualization

These charts are designed to make the evaluation story visible quickly:

- category score profile
- score distribution
- review-priority mix
- dataset/category performance


In [ ]:
# =============================================================================
# CELL 8: VISUALIZATION
# =============================================================================
display_answer_accuracy_visuals(run, display_tables, plan)


---

## 📝 9. Executive Report

The report below is rendered directly in the notebook for review. It is also saved as Markdown in the run folder.

For a more advanced future version, the deterministic report can be supplemented with optional LLM-generated narrative, clearly labeled as AI-generated analysis.


In [ ]:
# =============================================================================
# CELL 9: HTML EXECUTIVE REPORT
# =============================================================================
display_executive_report(run, plan, artifact_table)


---

## 💾 10. Save Artifacts

This section is the handoff checklist for reproducibility and reporting. It lists the generated evidence assets and the folder where each artifact is saved.

Use the HTML executive summary for leadership review, the Markdown technical report for source-control review, the raw outputs for row-level audit, the diagnostics file for exception review, and the checkpoint for resumable/replayable runs.


In [ ]:
# =============================================================================
# CELL 10: SAVE ARTIFACTS
# =============================================================================
saved_artifacts = display_saved_artifacts(run)


---

## 🧠 11. Interpretation and Next Steps

This notebook demonstrates the answer-accuracy slice of a broader capability evaluation suite.

Recommended next improvements:

1. Add exact multiple-choice grading for MMLU and ARC.
2. Add embedding similarity for open-ended factual QA.
3. Add optional judge narratives for flagged cases and executive reporting.
4. Add custom enterprise/domain golden sets.
5. Compare multiple target models under identical dataset selections.
6. Add regression testing against previous runs.

The mindset: evaluation is not just scoring. It is dataset design, metric design, diagnostic design, model-risk evidence, and communication.
